# v3 DPO Training: v3 from v2.1-lite + preference pairs

Trains DPO on top of v2.1-lite using the preference pairs from v3_pair_gen.
Exports v3 as GGUF q4_k_m for local evaluation.

**Problem targeted:** Problem 2 (behavioural misses on in-distribution data).
**Runs on:** Kaggle GPU (T4 or P100).
**Input:** `data/qwen3.5_latest/dpo_pairs.jsonl` (from v3_pair_gen output)
**Output:** `v3_dpo-qwen3b-q4_k_m.gguf`

Setup instructions before running:
1. Add v3_pair_gen output as input (contains dpo_pairs.jsonl).
2. Add v2-1-lite-multi-template-sft-unsloth output as input (contains merged_fp16_v2_1_lite).

## Cell 1: env check

In [1]:
import sys
print('Python:', sys.version)
!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tue Jun  2 09:16:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |             

## Cell 2: clone repo

In [2]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT  = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

Cloning into '/kaggle/working/repo'...


cloned -> /kaggle/working/repo
/kaggle/working/repo


## Cell 3: install dependencies

In [3]:
!pip install -q unsloth
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

import sys
sys.path.insert(0, '/kaggle/working/repo')
print('install done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 4: load preference pairs

In [4]:
import json
from pathlib import Path
from datasets import Dataset

# dpo_pairs.jsonl is saved to the repo data dir by v3_pair_gen,
# but on Kaggle it may arrive via a separate input dataset.
# Try repo data dir first, then /kaggle/input.
PAIRS_CANDIDATES = [
    Path('data/qwen3.5_latest/dpo_pairs.jsonl'),
    Path('/kaggle/input/v3-pair-gen/dpo_pairs.jsonl'),  # adjust name if needed
]

pairs_path = next((p for p in PAIRS_CANDIDATES if p.exists()), None)
assert pairs_path is not None, (
    "dpo_pairs.jsonl not found. Add v3_pair_gen output as a Kaggle input dataset."
)
print('loading pairs from', pairs_path)

with open(pairs_path) as f:
    pairs = [json.loads(l) for l in f if l.strip()]

# DPOTrainer needs prompt, chosen, rejected columns
ds = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in pairs
])

print(f'pairs loaded: {len(ds)}')

# Summary
from collections import Counter
modes = Counter(p['rejected_mode'] for p in pairs)
tmpls = Counter(p['template'] for p in pairs)
print('by template:', dict(tmpls))
print('by rejected_mode:', dict(modes))

loading pairs from data/qwen3.5_latest/dpo_pairs.jsonl
pairs loaded: 210
by template: {'referral_a': 41, 'mse': 99, 'soap': 70}
by rejected_mode: {'schema_invalid': 18, 'over_populated': 68, 'hallucination': 57, 'miss': 67}


## Cell 5: load v2.1 base + fresh LoRA

In [5]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

MODEL_PATH = "/kaggle/input/notebooks/nizamuddin/v2-1-lite-multi-template-sft-unsloth/merged_fp16_v2_1_lite"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('base loaded')

# Attach a fresh LoRA (separate from the SFT adapter — clean separation)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=0,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

base loaded


Unsloth 2026.5.10 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Cell 5b: Sequence-length audit

Check what fraction of pairs would be truncated at `max_seq_length=1024`.
This matches the SFT baseline for fair comparison. Any pair exceeding 900 tokens
(90% of limit) is flagged — review the count before training.

In [6]:
# Tokenise prompt+chosen and prompt+rejected to measure sequence lengths.
# DPOTrainer packs both into one forward pass; the longer of the two determines
# whether a pair is truncated.
# We check against max_seq_length=1024 (matching SFT baseline).
from pathlib import Path
import json
MAX_SEQ_LENGTH = 1024
WARN_THRESHOLD = 0.9  # flag pairs using >90% of budget
def load_jsonl(path: Path):
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)
## after generating pairs, they are uploaded to git, which is then dowloaded
pairs = list(load_jsonl(Path("/kaggle/working/repo/data/qwen3.5_latest/dpo_pairs.jsonl"))) 
print("pairs loaded:", len(pairs))
lengths = []
for p in pairs:
    combined_chosen   = p['prompt'] + p['chosen']
    combined_rejected = p['prompt'] + p['rejected']
    # Tokenise without adding special tokens to get raw length
    len_c = len(tokenizer.encode(combined_chosen,   add_special_tokens=False))
    len_r = len(tokenizer.encode(combined_rejected, add_special_tokens=False))
    lengths.append(max(len_c, len_r))

over_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
near_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH * WARN_THRESHOLD)
mean_len    = sum(lengths) / len(lengths)
max_len     = max(lengths)

print(f'pairs checked       : {len(lengths)}')
print(f'mean max-side length: {mean_len:.0f} tokens')
print(f'max  max-side length: {max_len} tokens')
print(f'pairs > {MAX_SEQ_LENGTH} (truncated): {over_limit}  ({100*over_limit/len(lengths):.1f}%)')
print(f'pairs > {int(MAX_SEQ_LENGTH*WARN_THRESHOLD)} (near limit): {near_limit}  ({100*near_limit/len(lengths):.1f}%)')

if over_limit > 0:
    print(f'\nWARNING: {over_limit} pairs will be silently truncated at {MAX_SEQ_LENGTH}.')
    print('These are consistent with the SFT training budget; no config change needed.')
    print('If >20% of pairs are truncated, consider raising max_seq_length and checking T4 OOM.')
else:
    print('\nAll pairs fit within max_seq_length=1024. No truncation risk.')


pairs loaded: 210
pairs checked       : 210
mean max-side length: 750 tokens
max  max-side length: 1183 tokens
pairs > 1024 (truncated): 5  (2.4%)
pairs > 921 (near limit): 13  (6.2%)

These are consistent with the SFT training budget; no config change needed.
If >20% of pairs are truncated, consider raising max_seq_length and checking T4 OOM.


## Cell 6: DPO training

In [7]:
from datasets import Dataset
ds = Dataset.from_list(
    [{"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]} for p in pairs]
)
# create train and val split
ds = ds.shuffle(seed=0)
split = ds.train_test_split(test_size=0.05, seed=0)  # ~10 eval pairs
## The signal we want is "is the model collapsing or diverging globally?" for this small dataset. we would stratify this in production case by template
train_ds = split["train"]
eval_ds  = split["test"]

In [8]:
from trl import DPOTrainer, DPOConfig

# Conservative hyperparameters: low beta keeps KL anchor tight,
# preventing the model from trading misses for hallucinations.
# Also, set keeping in mind running on T4 on kaggle

# steps_per_epoch = (len(ds)/per_device_train_batch_size)/gradient_accumulation_steps
# total_steps = num_train_epochs * steps_per_epoch
# warmup_steps = 0.1 * total_steps---so 5
config = DPOConfig(
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=10,
    per_device_eval_batch_size=2,
    lr_scheduler_type='cosine',
    warmup_steps=5,# see calc above
    seed=0,
    max_length=1024,
    max_prompt_length=700,
    output_dir='dpo_out_v3',
    logging_steps=10,
    save_strategy='no',
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

print('starting DPO training...')
trainer.train()
print('training complete')

Extracting prompt in train dataset (num_proc=8):   0%|          | 0/199 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=8):   0%|          | 0/199 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=8):   0%|          | 0/199 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=8):   0%|          | 0/11 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=8):   0%|          | 0/11 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 199 | Num Epochs = 2 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
10,0.692659,0.691501,0.002159,-0.000833,0.500000,0.002992,-26.931200,-25.103518,-2.219591,-2.164911
20,0.686632,0.688399,0.006433,-0.003570,0.666667,0.010003,-26.888464,-25.130880,-2.217698,-2.162771
30,0.673343,0.684830,0.012509,-0.005611,0.666667,0.018120,-26.827703,-25.151297,-2.216325,-2.161573
40,0.665041,0.682259,0.015322,-0.009177,0.666667,0.024499,-26.799576,-25.186953,-2.215419,-2.160471
50,0.654038,0.682204,0.016072,-0.008550,0.666667,0.024622,-26.792068,-25.180679,-2.215849,-2.160769


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

training complete


## Cell 7: sanity check before export

Run the trained model on a few in-distribution transcripts and verify
`schema_valid=1` on all outputs. If any fail, investigate before exporting.

In [9]:
from src.verifier import score_prediction
from src.prompts import build_inference_messages
from src.data_generation.templates import REGISTRY
import json

FastLanguageModel.for_inference(model)

DATA_ROOT = '/kaggle/working/repo/data/qwen3.5_latest' 

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


sanity_rows = []
for tpl, fname in [('soap', 'eval_in_dist.soap.jsonl'),
                   ('referral_a', 'eval_in_dist.referral_a.jsonl'),
                   ('mse', 'eval_in_dist.mse.jsonl')]:
    rows = load_jsonl(f'{DATA_ROOT}/{fname}')
    for r in rows[:2]: # Pick 2 samples per trained template for sanity check
        sanity_rows.append(dict(r, template=tpl))

print(f'sanity checking on {len(sanity_rows)} samples...')
print()
all_ok = True
for row in sanity_rows:
    template   = row['template']
    label_key  = REGISTRY[template]['label_key']
    gold       = row[label_key]
    transcript = row['transcript']

    messages = build_inference_messages(template, transcript)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, 
        max_new_tokens=1024, 
        do_sample=False,  # greedy
        #temperature=0.1, #cannot be 0.0
        pad_token_id=tokenizer.eos_token_id
    )
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    sc = score_prediction(template, raw, gold, transcript)
    status = 'OK' if sc['schema_valid'] else 'FAIL'
    if not sc['schema_valid']:
        all_ok = False
    print(f'  [{status}] template={template}  schema_valid={sc["schema_valid"]}  '
          f'overlap={sc["content_overlap"]:.2f}  wrong_null={sc["wrong_null"]}/{sc["gold_filled"]}')

print()
if all_ok:
    print('All sanity checks passed. Safe to export.')
else:
    print('WARNING: schema_valid=0 on some outputs. Investigate before exporting.')

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sanity checking on 6 samples...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [OK] template=soap  schema_valid=1  overlap=0.48  wrong_null=1/8


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [OK] template=referral_a  schema_valid=1  overlap=0.73  wrong_null=1/8


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [OK] template=referral_a  schema_valid=1  overlap=0.79  wrong_null=0/6


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [OK] template=mse  schema_valid=1  overlap=0.14  wrong_null=1/7
  [OK] template=mse  schema_valid=1  overlap=0.04  wrong_null=0/3

All sanity checks passed. Safe to export.


## Cell 8: merge LoRA and export GGUF q4_k_m

In [10]:
# Merge LoRA into the base weights
model.save_pretrained_merged('merged_fp16_v3', tokenizer, save_method='merged_16bit')
print('merged weights saved -> merged_fp16_v3/')

# Export to GGUF q4_k_m
# Unsloth names the file: v3_dpo-unsloth.Q4_K_M.gguf
model.save_pretrained_gguf('v3_dpo', tokenizer, quantization_method='q4_k_m')
print('GGUF saved')
print()
print('Download the GGUF and rename to: v3_dpo-qwen3b-q4_k_m.gguf')
print('Place in: models/ in the local repo')
!ls -lh *.gguf 2>/dev/null || echo 'no gguf in cwd, check Kaggle output panel'

Detected local model directory: /kaggle/input/notebooks/nizamuddin/v2-1-lite-multi-template-sft-unsloth/merged_fp16_v2_1_lite


Unsloth: Restored added_tokens_decoder metadata in merged_fp16_v3/tokenizer_config.json.


No existing and accessible Hugging Face cache directory found.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

Copied model.safetensors from local model directory
Splitting model.safetensors (size: 5.75 GB)...


...

No existing and accessible Hugging Face cache directory found.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

Copied model.safetensors from local model directory
Splitting model.safetensors (size: 5.75 GB)...


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:03<00:00, 15.88s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/kaggle/working/repo/v3_dpo`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['v3_dpo_gguf/merged_fp16_v2_1_lite.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This 